# 🌾 Crop Recommendation & Yield Prediction System — Exploratory Data Analysis & Modeling
### Bennett University | Machine Learning Subject Project — TEAM ROCKET
**Team Members:** Manan Gupta (S25CSEU0768), Vansh Agarwal (S25CSEU0743), Akasshat Garg (S25CSEU0755)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load Dataset & Inspection

In [ ]:
df = pd.read_csv('../data/crop_recommendation.csv')
df.columns = [c.strip().lower() for c in df.columns]
print("Dataset Shape:", df.shape)
df.head()

In [ ]:
df.info()
df.describe()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(9, 6))
sns.heatmap(df.drop(columns=['label']).corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Soil & Climate Parameters')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
features = ['n', 'p', 'k', 'temperature', 'humidity', 'ph', 'rainfall']
for i, feat in enumerate(features):
    r, c = divmod(i, 4)
    sns.histplot(df[feat], kde=True, ax=axes[r, c], color='teal')
    axes[r, c].set_title(f'Distribution: {feat.upper()}')
axes[1, 3].axis('off')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Train-Test Split

In [ ]:
X = df[['n', 'p', 'k', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 4. Model Training: Random Forest & XGBoost

In [ ]:
rf = RandomForestClassifier(n_estimators=200, criterion='entropy', max_depth=12, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

In [ ]:
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Random Forest Classifier')
plt.xlabel('Predicted Crop')
plt.ylabel('Actual Crop')
plt.show()